# Finalize Gene Clusters (manual merge)

## Inputs
- `results/clustering/candidates/{dataset}/candidate_clusters.tsv`  (from: workflow/scripts/clustering/generate_candidate_clusters.py)

## Outputs
- `resources/curated/final_clusters.tsv`  (manually curated — the 64→9 merge; consumed by enrichment.smk + ml.smk)

**This is a human-judgment step.** The two dicts below (`reformat_cluster`, `reorder_reformat_cluster`) were eyeballed from the HD candidate feature-space scatter. Re-run against the current dataset's candidates and adjust the mappings by looking at the review plot before writing `final_clusters.tsv`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parents[1]))  # repo root
from workflow.src.plotting.gene_level import visualize_cluster_on_feature_space

In [ ]:
# --- Config: point at the dataset whose candidates you are finalizing ---
DATASET = "HD_DIT_HAP_generationRAW"  # switch to your target dataset
CANDIDATES = Path.cwd().parents[1] / f"results/clustering/candidates/{DATASET}/candidate_clusters.tsv"
OUTPUT = Path.cwd().parents[1] / "resources/curated/final_clusters.tsv"

data_df = pd.read_csv(CANDIDATES, sep="\t", index_col=0)
data_df.shape, sorted(data_df['cluster'].unique())[:10]

## 1. Review the 64 candidate clusters

In [ ]:
fig = visualize_cluster_on_feature_space(data_df, 'cluster')
plt.show()

## 2. Merge 64 → intermediate, then renumber to 1..9

The dicts below are the HD starting point. **Adjust by eye** against the plot above for the current dataset — cluster geometry differs per dataset, so HD's mapping is unlikely to be correct for others.

In [ ]:
reformat_cluster = {
    0: 3, 1: 1, 2: 2, 3: 3, 4: 4, 5: 3, 6: 6, 7: 1, 8: 3, 9: 9,
    10: 3, 11: 11, 12: 1, 13: 13, 14: 11, 15: 4, 16: 4, 17: 1, 18: 3, 19: 1,
    20: 6, 21: 6, 22: 2, 23: 3, 24: 1, 25: 11, 26: 1, 27: 3, 28: 28, 29: 3,
    30: 28, 31: 11, 32: 1, 33: 11, 34: 13, 35: 3, 36: 4, 37: 9, 38: 4, 39: 3,
    40: 6, 41: 13, 42: 3, 43: 6, 44: 2, 45: 11, 46: 28, 47: 3, 48: 11, 49: 1,
    50: 1, 51: 28, 52: 9, 53: 3, 54: 1, 55: 1, 56: 3, 57: 3, 58: 6, 59: 3,
    60: 3, 61: 1, 62: 3, 63: 2,
}
data_df['revised_cluster'] = data_df['cluster'].map(reformat_cluster)

reorder_reformat_cluster = {6: 1, 2: 2, 9: 3, 13: 4, 28: 5, 1: 6, 4: 7, 11: 8, 3: 9}
data_df['revised_cluster'] = data_df['revised_cluster'].map(reorder_reformat_cluster)

data_df['revised_cluster'].value_counts().sort_index()

## 3. Review the merged 1..9 clusters (WT = 9)

In [ ]:
fig = visualize_cluster_on_feature_space(
    data_df, 'revised_cluster', show_box=True, legend=True, cluster_minus_one=True
)
plt.show()

## 4. Write the curated hub artifact

Only run this once you are satisfied with the merge above. This file is version-controlled and consumed by enrichment + ml.

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
data_df.to_csv(OUTPUT, sep="\t", index=True)
print(f"Wrote {len(data_df)} genes to {OUTPUT}")